In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt


In [ ]:
import numpy as np
from scipy.integrate import solve_ivp


def modelsys_2025_k16dynamic(p, retef, wntef, CCret, CCwnt, funcpercent, k8log, k17log, gamma1):
    # Description of Inputs
    # function [t, Sol] = nd_wnt_retinoid_hox_simp_updated(p, retef, wntef, CCret, CCwnt, funcpercent, apc_s, k8log, k17log, APCmin = min(Sol(:,19)))
    # p(end+1) = APCmin;

    # p = parameter vector for retinoid pathway
    # retef = ret on wnt param, wntef = wnt on ret param
    # RSS = RA SS value based on simplified pathway run with parameter set p and
    # link  # No longer used!
    # CCret = char conc for retinoid, CCwnt = char conc for wnt
    # funcpercent = apc functioning, apc_s = multiplier of axin synth
    # k8log = if k8 affected is T, k17log = if k17 affected is T

    ########################################################
    # Time discretization:
    N_t = 5e3
    t_0 = 0
    t_N = 30000
    time = np.linspace(t_0, t_N, int(N_t))

    #################Wnt Parameters##########################
    # Dimensional Parameters:
    DSH0 = 100
    TCF0 = 15
    APC0 = 100
    GSK0 = 50

    # Use logic input to determine if value of K8 or K17 change
    if k8log:
        K8 = (1 + (1 - funcpercent)) * 120
    else:
        K8 = 120

    if k17log:
        K17 = (1 + (1 - funcpercent)) * 1200
    else:
        K17 = 1200

    K16_max = 200 * gamma1  # 100 150
    K16_0 = 100  # 0.1
    Cs = 0.005  # 0.005
    nc = 2  # 2 3, 3.5 4.5
    K7 = 50
    # K16 = 30;
    K20 = 1
    K21 = 1
    Km = 98

    k1 = 0.182
    k2 = 1.82e-2
    k3 = 5e-2
    k4 = 0.267
    k5 = 0.133
    k6 = 9.09e-2
    k_6 = 0.909
    k9 = 206
    k10 = 206
    k11 = 0.417
    v12 = 0.423
    k13 = 2.57e-4
    v14 = (8.22e-5)  # *(1+0.01*apc_s); #increases synthesis of Axin by given percent
    k15 = .33

    # APC Regulating Function Parameters:
    Parameters = [212.8453, 39.9102, 34.1111]
    k19 = 1 / K17
    v18 = Parameters[0] * k19
    Kt = Parameters[1]
    Kb = Parameters[2]

    eta = 1
    if funcpercent < 1:
        v18 = v18 * (eta * funcpercent)

    ## Non-Dimensional Parameters:
    w = CCwnt  # conc. scaling for wnt going to be fed in like retinoid conc. scaling.
    gsk0 = GSK0 / w
    tcf0 = TCF0 / w
    dsh0 = DSH0 / w

    K7n = w / K7
    K8n = w / K8
    # K16n = K16*w;
    K17n = w / K17
    # K16p = K16/K17;
    K20n = w / K20
    # K21n = K21/K7;
    # Kmn  = Km/K17;
    Ktn = (TCF0 * w) / Kt
    Kbn = w / Kb

    k1n = k1 / k5
    k2n = k2 / k5
    k3n = k3 * w / k5
    k4n = k4 / k5
    k6n = (k6 * K21 * w ** 2) / (k5 * K7)
    k_6n = k_6 / k5
    k9n = (k9 * w) / (k5 * K8)
    k10n = k10 / k5
    # v10n = k10*k10n/k11;
    k11n = k11 / k5
    v12n = v12 / (w * k5)
    k13n = k13 / k5
    v14n = v14 / (k5 * w)
    # v14a = v14/K21/k5;
    # v14b = v14/K8/k5;
    # v14p = v14/K17/k5;
    # k14n = k9*v14b/k5;
    k15n = k15 * w / k5
    v18n = v18 / (w * k5)
    k19n = k19 / k5

    #################RAS Parameters###########################
    rkp1 = p[0]
    rkm1 = p[1]
    rk2 = p[2]
    rkp3 = p[3]
    rkm3 = p[4]
    rkp4 = p[5]
    rkm4 = p[6]
    rkp5 = p[7]
    rkm5 = p[8]
    rkp6 = p[9]
    rkm6 = p[10]
    rk7 = p[11]
    rk8 = p[12]
    rkp9 = p[13]
    rkm9 = p[14]
    rk10 = p[15]
    rk11 = 0
    rk12 = 0
    rk13 = 0
    rk14 = p[16]
    rk15 = p[17]
    rv16 = p[18]
    rv17 = p[19]
    rv18 = p[20]
    rv19 = p[21]
    rk20 = p[22]
    rk21 = p[23]
    rk22 = p[24]
    rk23 = p[25]
    MCsynth = p[26]

    ######################### HOX Parameters ##############################
    # Parameters from the HOX model
    k31 = 8.25 * 16.8182e2  # 0.12*1.6182e2 0.18  original = 0.25 2.25 30
    # k31 = 0.18*10.8182e2;
    # k31 = 0.12;
    k32 = 20.85 * 1.25e1  # 0.1    original 0.45*1.25e1 5.85
    k33 = 0.30 * 1.95e2  # 0.1, 0.2
    k34 = 0.25 * 10  # Can be tuned 0.25*10
    k36 = 0.13  # 0.15
    kp37 = 0.45 * 1.0e-1  # 0.45
    # kp37 = 0.45*5.0e-2; #0.45
    # kp37 = 0.25;
    km37 = 0.1 * 1.65  # 0.1*1.65 .5, 1.65
    # km37 = 0.09*1.65;#1.5, 1.65
    v31 = 0.15 * 3.333e1
    # v31 = 0.15;
    k38 = 0.083  # 0.10 0.09
    v32 = 0.20 * 3.333e1  # 0.20*3.333e1 2*3.333e1
    # v32 = 0.20;
    k39 = 0.15  # 0.15

    # k35 = 0.50; #0.10

    ###########

    ## HOXA5 deg, and MYC synth parameters
    a2 = 7.735
    # a3 = 1.5;7.5, 19.5;
    # a3 = 70.5;
    a3 = 50.5  # 50.5 45.555
    a4 = 1.5
    Kc = 1.0
    a1 = 55.5
    Kd = 0.01  # 20.5 0.005
    k33_max = 1.0  # 10 100 200 500
    n = 1  # 2 original 1
    km = 0.25  # 0.5 0.25

    ######################
    # New constants
    k = 1500
    a5 = 1
    a6 = 1
    n1 = 1  # 3
    n2 = 1  # 4
    n3 = 1
    knn1 = 200.5
    knn2 = 100.5

    phi = 0.00139

    k33 = lambda Nr: (k33_max * (w * Nr) ** n) / (Kd ** n + (w * Nr) ** n)  # new synth of HOXA13
    # Nondimensional parameters of HOX model scaled by Bcat
    k31n = k31 / (w * k5)
    k32n = k32 / k5
    k33n = lambda Nr: k33(Nr) / k5
    # k33n =  k33/ k5;
    k36n = k36 / k5
    kp37n = kp37 * w / k5
    km37n = km37 / k5
    v31n = v31 / (w * k5)
    k38n = k38 / k5
    v32n = v32 / (w * k5)
    k39n = k39 / k5
    # k35n = k35/k5;
    knn = k / (w * k5)

    # Linked parameters
    kp40 = 2.5 * 1e-6  # 6
    # kp40 = 3.6 * 1e-5;
    km40 = 1.0 * 1e-3  # 3 H13/beta blows up from 10^-5 and larger frequencies for 10^-1
    # km40 = 1.0 * 1e-6;
    k41 = 0.1475 * (1e-4)  # 0.15, 0.08 # Controls the oscillation and beta-catenin conc
    # k41 = 0.10*(1e-4); # w = 500

    # Added non-dim rate constant (linked parameters)
    kp40n = (kp40 * w) / k5
    # kp40n = kp40 / k5;
    km40n = km40 / k5  # this is important to H13/beta complex
    k41n = (k41 * w) / k5
    # k41n = k41 / k5 ;

    #############################################################
    # For NonDim purposes of Both pathways
    r = CCret  # choose char conc of RA
    a = 1 / k5  # choose any param that is 1/time, using k5 from wnt signaling pathway

    #############################################################
    # Linking Pathways: WNT on RAS (incr. CYP synthesis)
    # bctf = @(x) TCF0.*x./(K16+w*x); #formula for conc. of bcat/tcf at any time #MAKE SURE NONDIM!
    # bctf = @(x, Ci) TCF0 .* x ./ (K16_max * (1 + ((w*Ci).^nc) ./ (Cs.^nc + (w*Ci).^nc)) + w * x);
    bctf = lambda x, Ci: TCF0 * x / (K16_0 + (K16_max * ((w * Ci) ** nc)) / (Cs ** nc + ((w * Ci) ** nc)) + w * x)
    # cyp_synthFunc = @ (x,y) (rv18 + wntef*bctf(y) + MCsynth*(r*x).^2)./(1+(r*x).^2); #is written as a function of (RA, B-catenin)
    cyp_synthFunc = lambda x, y, Ci: (rv18 + wntef * bctf(y, Ci) + MCsynth * (r * x) ** 2) / (1 + (r * x) ** 2)

    # Define a linked function to get RA link on WNT to match expected results.
    # This new link is through APC degradation instead synth as before.
    Pmin = 0.002
    APCdeg = lambda x, y: (k19n / (1 + retef * r * x ** 2)) * (y >= 0.75 * Pmin) + (k19n * (1 + retef * r * x ** 2)) * (y < 0.75 * Pmin)
    # K16n = (w*K16)/(1+retef*RSS);
    gamma = 0.025  # gamma = 0.005;#0.05 # original 0.025

    # b= 150;

    ###############################################################
    # Sources for Both Pathways
    # W = @(t) 1.*(t>0); #Wnt always on
    W = lambda t: 0. * (t < 3000) * (t > 25000) + 3 * 1. * (t >= 3000) * (t <= 25000)  # wnt off

    # #Periodic source for RAS pathway
    A = 1e2  # amplitude
    B = np.pi / 6  # controls period pi/6 = 12hrs
    C = 0  # controls phase shift
    D = 2e2
    # gtild = @(t) (a/r)*(A + D*cos(B*a*t - C));
    gtild = lambda t: (a / r) * A * (1 + np.cos(B * a * t - C))
    # #ftild = @(t) 0.1; #Bs = pi/100; #gtild_s = @(t)(a/r)*A*(1 + cos(Bs*a*t - C));
    # #
    # #     # A = 0.0753*1e1;
    # #     # B = 0.03*1e1;
    #     A = 600;
    #     B = 400;
    #     C = pi/6;
    #     T = 1; # Period in hours
    #     phi = 0; # Phase shift
    #     gtild = @(t) (a/r)*(A + B * sin((C* a * t) + phi));

    # Parameters for the Generalized Gaussian function
    # A = 1;        # Peak amplitude
    # mu = 4500;      # Center of the peak (time of maximum effect)
    # sigma = 200;    # Width of the bell
    # p = 2;        # Shape parameter (2 for standard Gaussian)
    #
    # # Define the Generalized Gaussian function
    # f = @(t) (a/r) * A * exp(-((abs(t - mu) / sigma).^p));
    # treatvals = f(time);

    # # Parameters for the logistic-based bell-shaped function
    # L1 = 1.5;        # Amplitude of the rising phase
    # L2 = 2;        # Amplitude of the falling phase
    # t1 = 3000;       # Time when rise begins
    # t2 = 4000;       # Time when fall begins
    # k1 = 0.1;      # Growth rate for the rising phase
    # k2 = 0.2;      # Decay rate for the falling phase
    #
    #
    # # Define the function
    # f = @(t) (a/r) * (L1 ./ (1 + exp(-k1 * (t - t1)))) - (L2 ./ (1 + exp(-k2 * (t - t2))));
    #
    # treatvals = f(time);

    # s = 1e3;
    # c = 100;
    # treat = @ (t) (a/r).*s.*exp((-(t-4500).^2)./(2*c^2));
    # treatvals = treat(time);

    # treatvals = treat(time);

    t_start = 10000  # Treatment starts
    t_end = 15000  # Treatment ends
    dose_strength = 0e2  # Scaled-up dose strength 1.2*2.5e2
    q = 100  # Smoothness factor 100-300

    # Smooth treatment function using tanh
    treat = lambda t: (a / r) * (dose_strength / 2) * \
        (np.tanh((t - t_start) / q) - np.tanh((t - t_end) / q))

    treatvals = treat(time)

    ###############################################################
    # Define System
    #######################################################################
    # Define the explicit differential equations for WNT:
    dVdtn = lambda t, V, Di, Db, Bp, Da, P, Ba, X: k1n * (dsh0 - V) * W(t) - k2n * V

    dDidtn = lambda t, V, Di, Db, Bp, Da, P, Ba, X: -(k3n * V + k4n + k_6n) * Di + Da + (k6n * P * X *
                                                                                        (gsk0 - (1 + K8n * Ba) * Da - Di - Db)) / (K21 + w * X)

    dDbdtn = lambda t, V, Di, Db, Bp, Da, P, Ba, X: k9n * Da * Ba - k10n * Db

    dBpdtn = lambda t, V, Di, Db, Bp, Da, P, Ba, X: k10n * Db - k11n * Bp

    # Define the implicit differential equations:
    An = lambda t, V, Di, Db, Bp, Da, P, Ba, X: -K8n * (K8 + w * Ba) * X / (K21 + w * X)
    Bn = lambda t, V, Di, Db, Bp, Da, P, Ba, X: K7n * X
    Cn = lambda t, V, Di, Db, Bp, Da, P, Ba, X: K20n * X - w * K8n * Da * X / (K21 + w * X)
    Dn = lambda t, V, Di, Db, Bp, Da, P, Ba, X: 1 + K7n * P + K20n * Ba + K21 * w * \
        (gsk0 - (1 + K8n * Ba) * Da - Di - Db) / ((K21 + w * X) ** 2)
    En = lambda t, V, Di, Db, Bp, Da, P, Ba, X: (1 + K8n * Ba)
    Fn = lambda t, V, Di, Db, Bp, Da, P, Ba, X: K8n * Da
    Gn = lambda t, V, Di, Db, Bp, Da, P, Ba, X: K8n * Ba
    Hn = lambda t, V, Di, Db, Bp, Da, P, Ba, X: K17n * Ba
    # In = @(t,V,Di,Db,Bp,Da,P,Ba,X) 1+K8n*Da+K16n*tcf0./((K16+w*Ba).^2)+K17n*P+K20n*X;
    In = lambda t, V, Di, Db, Bp, Da, P, Ba, X, Ci: \
        1 + K8n * Da + ((K16_0 + (K16_max * ((w * Ci) ** nc)) / (Cs ** nc + ((w * Ci) ** nc))) * tcf0) / ((K16_0 + (K16_max * ((w * Ci) ** nc)) / (Cs ** nc + ((w * Ci) ** nc)) + w * Ba) ** 2) + K17n * P + K20n * X

    Jn = lambda t, V, Di, Db, Bp, Da, P, Ba, X: K20n * Ba
    Kn = lambda t, V, Di, Db, Bp, Da, P, Ba, X: 1 + K8n * Ba
    Ln = lambda t, V, Di, Db, Bp, Da, P, Ba, X: 1 + K7n * X + K17n * Ba
    Mn = lambda t, V, Di, Db, Bp, Da, P, Ba, X: K8n * Da + K17n * P
    Nn = lambda t, V, Di, Db, Bp, Da, P, Ba, X: K7n * P

    RHS1n = lambda t, V, Di, Db, Bp, Da, P, Ba, X, Ci: v14n + (k3n * V + k_6n) * Di - \
        k6n * P * X * \
        (gsk0 - (1 + K8n * Ba) * Da - Di - Db) / (K21 + w * X) - \
        k15n * P * X / (Km + w * P) + w * X / (K21 + w * X) * \
        (dDidtn(t, V, Di, Db, Bp, Da, P, Ba, X) +
         dDbdtn(t, V, Di, Db, Bp, Da, P, Ba, X))

    RHS2n = lambda t, V, Di, Db, Bp, Da, P, Ba, X: k4n * Di - (1 + k9n * Ba) * Da + k10n * Db

    # RHS3n = @(t,V,Di,Db,Bp,Da,P,Ba,X) v12n-(k13n+k9n*Da).*Ba; No change made
    RHS3n = lambda t, V, Di, Db, Bp, Da, P, Ba, X, H5, H13, Ci: v12n - (k13n + k9n * Da + kp40n * H13 +
                                                                        k41n * H5) * Ba + km40n * Ci  # Ba further reduced by HOX proteins

    # RHS4n = @(t,V,Di,Db,Bp,Da,P,Ba,X,R,H13,Ci) v18n./(1+Ktn*Ba./((30 + ((K16_max - 30)*w*Ci) / (w*Ci + Cs))+w*Ba)+Kbn*Ba + gamma*w*H13)-... #addition here for RA and H13 feedback on APC synth
    #                                   APCdeg(R, P)*P-...
    #                                   (dDidtn(t,V,Di,Db,Bp,Da,P,Ba,X)+...
    #                                   dDbdtn(t,V,Di,Db,Bp,Da,P,Ba,X));

    RHS4n = lambda t, V, Di, Db, Bp, Da, P, Ba, X, R, H13, Ci: v18n / (1 + Ktn * Ba / ((K16_0 + (K16_max * ((w * Ci) ** nc)) / (Cs ** nc + ((w * Ci) ** nc))) + w * Ba) + Kbn * Ba + gamma * w * H13) - \
        APCdeg(R, P) * P - \
        (dDidtn(t, V, Di, Db, Bp, Da, P, Ba, X) +
         dDbdtn(t, V, Di, Db, Bp, Da, P, Ba, X))  # addition here for RA and H13 feedback on APC synth

    # RHS4n = @(t,V,Di,Db,Bp,Da,P,Ba,X,R,H5,H13) b*H5*v18n./(1+Ktn*Ba./(K16+w*Ba)+Kbn*Ba + gamma*w*H13)-... #addition here for RA and H13 feedback on APC synth
    #                                   APCdeg(R, P)*P-...
    #                                   (dDidtn(t,V,Di,Db,Bp,Da,P,Ba,X)+...
    #                                   dDbdtn(t,V,Di,Db,Bp,Da,P,Ba,X));

    # Define the Matrix and Vector:
    Matrixn = lambda t, V, Di, Db, Bp, Da, P, Ba, X, Ci: np.array([[An(t, V, Di, Db, Bp, Da, P, Ba, X),
                                                                    Bn(t, V, Di, Db, Bp, Da, P, Ba, X),
                                                                    Cn(t, V, Di, Db, Bp, Da, P, Ba, X),
                                                                    Dn(t, V, Di, Db, Bp, Da, P, Ba, X)],
                                                                   [En(t, V, Di, Db, Bp, Da, P, Ba, X),
                                                                    0,
                                                                    Fn(t, V, Di, Db, Bp, Da, P, Ba, X),
                                                                    0],
                                                                   [Gn(t, V, Di, Db, Bp, Da, P, Ba, X),
                                                                    Hn(t, V, Di, Db, Bp, Da, P, Ba, X),
                                                                    In(t, V, Di, Db, Bp, Da, P, Ba, X, Ci),
                                                                    Jn(t, V, Di, Db, Bp, Da, P, Ba, X)],
                                                                   [Kn(t, V, Di, Db, Bp, Da, P, Ba, X),
                                                                    Ln(t, V, Di, Db, Bp, Da, P, Ba, X),
                                                                    Mn(t, V, Di, Db, Bp, Da, P, Ba, X),
                                                                    Nn(t, V, Di, Db, Bp, Da, P, Ba, X)]])

    Vectorn = lambda t, V, Di, Db, Bp, Da, P, Ba, X, R, H5, H13, Ci: np.array([RHS1n(t, V, Di, Db, Bp, Da, P, Ba, X, Ci),
                                                                               RHS2n(t, V, Di, Db, Bp, Da, P, Ba, X),
                                                                               RHS3n(t, V, Di, Db, Bp, Da, P, Ba, X, H5, H13, Ci),
                                                                               RHS4n(t, V, Di, Db, Bp, Da, P, Ba, X, R, H13, Ci)])

    # Vectorn = @(t,V,Di,Db,Bp,Da,P,Ba,X,R,H5,H13,Ci) [RHS1n(t,V,Di,Db,Bp,Da,P,Ba,X,Ci);
    #                                      RHS2n(t,V,Di,Db,Bp,Da,P,Ba,X);
    #                                      RHS3n(t,V,Di,Db,Bp,Da,P,Ba,X,H5,H13,Ci);
    #                                      RHS4n(t,V,Di,Db,Bp,Da,P,Ba,X,R,H5,H13)];

    # Solutionn = @(t,V,Di,Db,Bp,Da,P,Ba,X,R,H5,H13,Ci) Matrixn(t,V,Di,Db,Bp,Da,P,Ba,X)\...
    #                                       Vectorn(t,V,Di,Db,Bp,Da,P,Ba,X,R,H5,H13,Ci);
    #
    #
    # Solutionn = @(t,V,Di,Db,Bp,Da,P,Ba,X,R,H5,H13,Ci) ...
    #     Matrixn(t,V,Di,Db,Bp,Da,P,Ba,X) \ ...
    #     Vectorn(t,V,Di,Db,Bp,Da,P,Ba,X,R,H5,H13,Ci);

    Solutionn = lambda t, V, Di, Db, Bp, Da, P, Ba, X, R, H5, H13, Ci: \
        np.linalg.solve(Matrixn(t, V, Di, Db, Bp, Da, P, Ba, X, Ci), Vectorn(t, V, Di, Db, Bp, Da, P, Ba, X, R, H5, H13, Ci))

    # Set up RAS Pathway Eqns

    dRodtn = lambda t, Ro, Ra, A, R, B, Br, N, Nr, C, Cr, Dc, Dn, Bc: a * (rkm1 * Ra - rkp1 * Ro) + gtild(t)
    dRadtn = lambda t, Ro, Ra, A, R, B, Br, N, Nr, C, Cr, Dc, Dn, Bc: a * (rkp1 * Ro - (rkm1 + rk2 * r * A) * Ra)
    dAdtn = lambda t, Ro, Ra, A, R, B, Br, N, Nr, C, Cr, Dc, Dn, Bc: (a / r) * rv19 - a * (rk23 * A)
    dRdtn = lambda t, Ro, Ra, A, R, B, Br, N, Nr, C, Cr, Dc, Dn, Bc: treat(t) + a * (rk2 * r * Ra * A - rkp3 * r * R * B + (rkm3 + rk14) * Br - rkp4 * r * R * N + (rkm4 + rk15) * Nr - rkp5 * r * R * C + rkm5 * Cr)
    dBdtn = lambda t, Ro, Ra, A, R, B, Br, N, Nr, C, Cr, Dc, Dn, Bc: (a / r) * rv16 + a * (-(rk20 + rkp3 * r * R) * B + rkm3 * Br + rk10 * Dn)
    dBrdtn = lambda t, Ro, Ra, A, R, B, Br, N, Nr, C, Cr, Dc, Dn, Bc: a * (rkp3 * r * R * B - rkm3 * Br - rkp6 * r * Br * C + rkm6 * Dc - rkp9 * r * Br * N + rkm9 * Dn - rk14 * Br)
    dNdtn = lambda t, Ro, Ra, A, R, B, Br, N, Nr, C, Cr, Dc, Dn, Bc: (a / r) * rv17 + a * (-(rk21 + rkp4 * r * R) * N + rkm4 * Nr - rkp9 * r * Br * N + rkm9 * Dn)
    dNrdtn = lambda t, Ro, Ra, A, R, B, Br, N, Nr, C, Cr, Dc, Dn, Bc: a * (rkp4 * r * R * N - (rkm4 + rk15) * Nr + rk10 * Dn)
    # dCdtn  = @(t,Ro,Ra,A,R,B,Br,N,Nr,C,Cr,Dc,Dn,Bc,Ba) (a/r)*cyp_synthFunc(R,Ba)+ a*(-(rk22+rkp5*r.*R).*C + (rkm5+rk8).*Cr - rkp6*r.*Br.*C + rkm6.*Dc);
    dCdtn = lambda t, Ro, Ra, A, R, B, Br, N, Nr, C, Cr, Dc, Dn, Bc, Ba, Ci: \
        (a / r) * cyp_synthFunc(R, Ba, Ci) + a * (-(rk22 + rkp5 * r * R) * C + (rkm5 + rk8) * Cr - rkp6 * r * Br * C + rkm6 * Dc)
    dCrdtn = lambda t, Ro, Ra, A, R, B, Br, N, Nr, C, Cr, Dc, Dn, Bc: a * (rkp5 * r * R * C - (rkm5 + rk8) * Cr)
    dDcdtn = lambda t, Ro, Ra, A, R, B, Br, N, Nr, C, Cr, Dc, Dn, Bc: a * (rkp6 * r * Br * C - (rkm6 + rk7) * Dc)
    dDndtn = lambda t, Ro, Ra, A, R, B, Br, N, Nr, C, Cr, Dc, Dn, Bc: a * (rkp9 * r * Br * N - (rkm9 + rk10) * Dn)
    dBcdtn = lambda t, Ro, Ra, A, R, B, Br, N, Nr, C, Cr, Dc, Dn, Bc: a * rk7 * Dc

    # u14 = tcf0 * Ba ./ (1 + Ba); # Bcat/TCF replaced by bctf
    # k33 = @(Nr) (k33_max *(w*Nr).^n)/(Kd^n + (w*Nr).^n); # new synth of HOXA13
    # k33n = @(Nr) (Kd+a1*(w*Nr).^n)/(1 + (w*Nr).^n);
    k35n = lambda Ca: (k34 + a3 * (w * Ca) ** n3) / (1 + (phi * w * Ca) ** n3) / k5  # Deg of HOXA5 by MYC/MIZ1 complex
    v30n = lambda bctf: (Kc + a2 * w * bctf) / (1 + w * bctf) / (w * k5)  # MYC synth
    # v32n = @(Ba) (Kd + a4 * K16 * Ba)/(1 + K16 * Ba); # new synth of HOXA13

    # dH5dtn  = @(t,H5,M,Mi,Ca,H13,Ci,Nr,Ba) knn*((w*Mi).^n1/(a5.^n1 +(w*Mi).^n1)).*((w*Nr).^n2/(a6.^n2 +(w*Nr).^n2)) - k35n(Ca).*H5;

    # dH5dtn  = @(t,H5,M,Mi,Ca,H13,Ci,Nr,Ba) knn*((w*Mi).^n1/(a5.^n1 +(w*Mi).^n1)).*((w*Nr).^n2/(a6.^n2 +(w*Nr).^n2)) - k35n*(k34 + a3*w*Ca).*H5;
    # dH5dtn  = @(t,H5,M,Mi,Ca,H13,Ci,Nr,Ba) k31n + k32n*Mi + k33n*Nr - k35n*(k34 + a3*w*Ca).*H5;

    # Differential equations of the HOX model
    # dH5dtn  = @(t,H5,M,Mi,Ca,H13,Ci,Nr,Ba) k31n + (k32n*(w*Mi).^n1)/(a5.^n1 +(w*Mi).^n1) + (k33n*(w*Nr).^n2)/(a6.^n2 +(w*Nr).^n2) - k35n(Ca).*H5;
    dH5dtn = lambda t, H5, M, Mi, Ca, H13, Ci, Nr, Ba: k31n + k32n * Mi + k33n(Nr) * Nr - k35n(Ca) * H5
    # dH5dtn  = @(t,H5,M,Mi,Ca,H13,Ci,Nr,Ba) k31n + k32n*Mi + k33n*Nr - k35n(Ca).*H5;
    # dMdtn   = @(t,H5,M,Mi,Ca,H13,Ci,Nr,Ba) v30n(bctf(Ba)) - k36n.* M - kp37n.* M.*Mi + km37n.*Ca;
    dMdtn = lambda t, H5, M, Mi, Ca, H13, Ci, Nr, Ba: v30n(bctf(Ba, Ci)) - k36n * M - kp37n * M * Mi + km37n * Ca
    dMidtn = lambda t, H5, M, Mi, Ca, H13, Ci, Nr, Ba: v31n - k38n * Mi - kp37n * M * Mi + km37n * Ca
    dCadt = lambda t, H5, M, Mi, Ca, H13, Ci, Nr, Ba: kp37n * M * Mi - km37n * Ca
    dH13dtn = lambda t, H5, M, Mi, Ca, H13, Ci, Nr, Ba: v32n - k39n * H13 - kp40n * H13 * Ba + km40n * Ci
    # dH13dtn = @(t,H5,M,Mi,Ca,H13,Ci,Nr,Ba) v32n(Ba) - k39n.*H13 - kp40n.*H13.*Ba + km40n.*Ci;
    dCidtn = lambda t, H5, M, Mi, Ca, H13, Ci, Nr, Ba: kp40n * H13 * Ba - km40n * Ci

    def RHSn(t, U):
        return np.array([
            dRodtn(t, U[0], U[1], U[2], U[3], U[4], U[5], U[6], U[7], U[8], U[9], U[10], U[11], U[12]),
            dRadtn(t, U[0], U[1], U[2], U[3], U[4], U[5], U[6], U[7], U[8], U[9], U[10], U[11], U[12]),
            dAdtn(t, U[0], U[1], U[2], U[3], U[4], U[5], U[6], U[7], U[8], U[9], U[10], U[11], U[12]),
            dRdtn(t, U[0], U[1], U[2], U[3], U[4], U[5], U[6], U[7], U[8], U[9], U[10], U[11], U[12]),
            dBdtn(t, U[0], U[1], U[2], U[3], U[4], U[5], U[6], U[7], U[8], U[9], U[10], U[11], U[12]),
            dBrdtn(t, U[0], U[1], U[2], U[3], U[4], U[5], U[6], U[7], U[8], U[9], U[10], U[11], U[12]),
            dNdtn(t, U[0], U[1], U[2], U[3], U[4], U[5], U[6], U[7], U[8], U[9], U[10], U[11], U[12]),
            dNrdtn(t, U[0], U[1], U[2], U[3], U[4], U[5], U[6], U[7], U[8], U[9], U[10], U[11], U[12]),
            dCdtn(t, U[0], U[1], U[2], U[3], U[4], U[5], U[6], U[7], U[8], U[9], U[10], U[11], U[12], U[19], U[26]),
            dCrdtn(t, U[0], U[1], U[2], U[3], U[4], U[5], U[6], U[7], U[8], U[9], U[10], U[11], U[12]),
            dDcdtn(t, U[0], U[1], U[2], U[3], U[4], U[5], U[6], U[7], U[8], U[9], U[10], U[11], U[12]),
            dDndtn(t, U[0], U[1], U[2], U[3], U[4], U[5], U[6], U[7], U[8], U[9], U[10], U[11], U[12]),
            dBcdtn(t, U[0], U[1], U[2], U[3], U[4], U[5], U[6], U[7], U[8], U[9], U[10], U[11], U[12]),
            dVdtn(t, U[13], U[14], U[15], U[16], U[17], U[18], U[19], U[20]),
            dDidtn(t, U[13], U[14], U[15], U[16], U[17], U[18], U[19], U[20]),
            dDbdtn(t, U[13], U[14], U[15], U[16], U[17], U[18], U[19], U[20]),
            dBpdtn(t, U[13], U[14], U[15], U[16], U[17], U[18], U[19], U[20]),
            *Solutionn(t, U[13], U[14], U[15], U[16], U[17], U[18], U[19], U[20], U[3], U[21], U[25], U[26]),
            dH5dtn(t, U[21], U[22], U[23], U[24], U[25], U[26], U[7], U[19]),
            dMdtn(t, U[21], U[22], U[23], U[24], U[25], U[26], U[7], U[19]),
            dMidtn(t, U[21], U[22], U[23], U[24], U[25], U[26], U[7], U[19]),
            dCadt(t, U[21], U[22], U[23], U[24], U[25], U[26], U[7], U[19]),
            dH13dtn(t, U[21], U[22], U[23], U[24], U[25], U[26], U[7], U[19]),
            dCidtn(t, U[21], U[22], U[23], U[24], U[25], U[26], U[7], U[19])
        ])  # Ensure Column Vector

    # Solutionn(t,U(14),U(15),U(16),U(17),U(18),U(19),U(20),U(21),U(4));

    #######################################################################
    # Initial Conditions from Katie:
    V_0 = 0
    Di_0 = 4.83e-3
    Db_0 = 2.02e-3
    Bp_0 = 1
    Da_0 = 9.66e-3
    P_0 = 18.116
    # P_0 = APC0/K17; #allow decrease or increase in APC IC
    Ba_0 = 25.1
    X_0 = 4.93e-4

    W_0n = (1 / w) * np.array([V_0, Di_0, Db_0, Bp_0, Da_0, P_0, Ba_0, X_0])

    # Initial conditions:
    x1_0 = 10
    x2_0 = 10
    x3_0 = 10
    x4_0 = 100  # RA
    x5_0 = 1
    x6_0 = 0
    x7_0 = 1
    x8_0 = 0
    x9_0 = 0.01
    x10_0 = 0
    x11_0 = 0
    x12_0 = 0
    x13_0 = 0

    R_0n = (1 / 1000) * np.array([x1_0, x2_0, x3_0, x4_0, x5_0, x6_0, x7_0, x8_0, x9_0, x10_0,
                                  x11_0, x12_0, x13_0])

    ###################################################
    # # ##Main Test For w =1000
    # H5_0 = 0.00003;#0.014;#0.0115; # 0.00865, 0.00835, 10.0; 26.0; 130.0, 128.0729, 130.0729
    # H5_0 = 0.0033;
    H5_0 = 0.00146
    # M_0 = 0.050; #1.5227; 1.5 0.045
    # M_0 = 0.055;
    M_0 = 0.04999
    # Mi_0 = 0.057; # 0.049, 0.055
    # Mi_0 = 0.058;
    Mi_0 = 0.0601
    # Ca_0 = 2.25; #5.7101, 5.0; 0.0
    # Ca_0 = 0.8984; # 0.70 0.75
    Ca_0 = 0.84028
    H13_0 = 0.04439  # 0.0333 0.0439
    Ci_0 = 0.0012  # 1.8901; 0

    H_0n = np.array([H5_0, M_0, Mi_0, Ca_0, H13_0, Ci_0])

    U_0n = np.concatenate([R_0n, W_0n, H_0n])
    #######################################################################
    # Solver:
    sol = solve_ivp(RHSn, (t_0, t_N), U_0n, method='BDF', t_eval=time)
    print(sol.success, sol.message)
    print(np.max(np.abs(sol.y)))
    t = sol.t
    U = sol.y.T
    # BcatTcf = bctf(U(:,20));
    # Sol = [U BcatTcf];
    BcatTcf = bctf(U[:, 19], U[:, 26])

    Sol = np.column_stack([U, BcatTcf])

    return t, Sol, treatvals


if __name__ == "__main__":
    import numpy as np

    # Real 27-element RA parameter vector (from the driver script)
    p = np.array([
        0.5, 0.1, 0.5, (2.13e11)*(1e-9), 1.32e1, (6.00e9)*(1e-0), 3.60e1,
        (1.2e11)*(1e-9), 4.00e1, 1.01e1, 1.01e1, 1.02e1, 2.70e1,
        (3.60e9)*(1e-7), 3.00e1, 1.00e-1, 3.85e-2, 1.73e-3, 2.00e-1,
        2.00e-1, 2.00e-2, 2.50e-2, 3.85e-2, 1.73e-1, 8.35e-3, 1.00e-2, 1.00e-1,
    ])

    t, Sol, treatvals = modelsys_2025_k16dynamic(
        p           = p,
        retef       = 0.0015,
        wntef       = 100,
        CCret       = 1000.0,
        CCwnt       = 1000.0,
        funcpercent = 1.0,
        k8log       = False,
        k17log      = False,
        gamma1      = 1.0,
    )

    print("Shape of Sol:", Sol.shape)
    print("Time points:", len(t))

In [ ]:
def modelsys_2025(p, retef, wntef, CCret, CCwnt, funcpercent, apc_s, k8log, k17log):
    """Constant-K16 coupled WNT/RAS/HOX model (K16 = 30)."""
    N_t = 5e3
    t_0 = 0
    t_N = 30000
    time = np.linspace(t_0, t_N, int(N_t))

    DSH0 = 100; TCF0 = 15; APC0 = 100; GSK0 = 50

    if k8log:
        K8 = (1 + (1 - funcpercent)) * 120
    else:
        K8 = 120
    if k17log:
        K17 = (1 + (1 - funcpercent)) * 1200
    else:
        K17 = 1200

    K16 = 30   # constant
    K7 = 50; K20 = 1; K21 = 1; Km = 98

    k1 = 0.182; k2 = 1.82e-2; k3 = 5e-2; k4 = 0.267; k5 = 0.133
    k6 = 9.09e-2; k_6 = 0.909; k9 = 206; k10 = 206; k11 = 0.417
    v12 = 0.423; k13 = 2.57e-4
    v14 = (8.22e-5) * (1 + 0.01 * apc_s)
    k15 = 0.33

    Parameters = [212.8453, 39.9102, 34.1111]
    k19 = 1 / K17
    v18 = Parameters[0] * k19
    Kt = Parameters[1]; Kb = Parameters[2]

    eta = 1
    if funcpercent < 1:
        v18 = v18 * (eta * funcpercent)

    w = CCwnt
    gsk0 = GSK0 / w; tcf0 = TCF0 / w; dsh0 = DSH0 / w

    K7n = w / K7; K8n = w / K8; K17n = w / K17; K20n = w / K20
    Ktn = (TCF0 * w) / Kt; Kbn = w / Kb

    k1n = k1 / k5; k2n = k2 / k5; k3n = k3 * w / k5; k4n = k4 / k5
    k6n = (k6 * K21 * w**2) / (k5 * K7); k_6n = k_6 / k5
    k9n = (k9 * w) / (k5 * K8); k10n = k10 / k5; k11n = k11 / k5
    v12n = v12 / (w * k5); k13n = k13 / k5; v14n = v14 / (k5 * w)
    k15n = k15 * w / k5; v18n = v18 / (w * k5); k19n = k19 / k5

    rkp1=p[0]; rkm1=p[1]; rk2=p[2]; rkp3=p[3]; rkm3=p[4]
    rkp4=p[5]; rkm4=p[6]; rkp5=p[7]; rkm5=p[8]
    rkp6=p[9]; rkm6=p[10]; rk7=p[11]; rk8=p[12]
    rkp9=p[13]; rkm9=p[14]; rk10=p[15]
    rk11=0; rk12=0; rk13=0
    rk14=p[16]; rk15=p[17]; rv16=p[18]; rv17=p[19]
    rv18=p[20]; rv19=p[21]; rk20=p[22]; rk21=p[23]
    rk22=p[24]; rk23=p[25]; MCsynth=p[26]

    k31=8.25*16.8182e2; k32=20.85*1.25e1; k34=0.25*10
    k36=0.13; kp37=0.45*1.0e-1; km37=0.1*1.65
    v31=0.15*3.333e1; k38=0.083; v32=0.20*3.333e1; k39=0.15
    a2=7.735; a3=50.5; Kc=1.0; Kd=0.01
    k33_max=1.0; n=1; phi=0.00139

    k33 = lambda Nr: (k33_max * (w * Nr)**n) / (Kd**n + (w * Nr)**n)
    k31n = k31 / (w * k5); k32n = k32 / k5
    k33n = lambda Nr: k33(Nr) / k5
    k36n = k36 / k5; kp37n = kp37 * w / k5; km37n = km37 / k5
    v31n = v31 / (w * k5); k38n = k38 / k5
    v32n = v32 / (w * k5); k39n = k39 / k5

    kp40=2.5e-6; km40=1.0e-3; k41=0.1475e-4
    kp40n = (kp40 * w) / k5; km40n = km40 / k5; k41n = (k41 * w) / k5

    r = CCret
    a = 1 / k5

    # Constant-K16 bctf (no Ci dependence)
    bctf = lambda x: TCF0 * x / (K16 + w * x)
    cyp_synthFunc = lambda x, y: (rv18 + wntef * bctf(y) + MCsynth * (r * x)**2) / (1 + (r * x)**2)

    Pmin = 0.002
    APCdeg = lambda x, y: (
        (k19n / (1 + retef * r * x**2)) * (y >= 0.75 * Pmin)
        + (k19n * (1 + retef * r * x**2)) * (y < 0.75 * Pmin)
    )
    gamma = 0.025

    W = lambda t: 3.0 * (t >= 3000) * (t <= 25000)

    A_amp = 1e2; B = np.pi / 6; C = 0
    gtild = lambda t: (a / r) * A_amp * (1 + np.cos(B * a * t - C))

    t_start=10000; t_end=15000; dose_strength=0e2; q=100
    treat = lambda t: (a/r) * (dose_strength/2) * (np.tanh((t-t_start)/q) - np.tanh((t-t_end)/q))
    treatvals = treat(time)

    # WNT ODEs
    dVdtn  = lambda t,V,Di,Db,Bp,Da,P,Ba,X: k1n*(dsh0-V)*W(t) - k2n*V
    dDidtn = lambda t,V,Di,Db,Bp,Da,P,Ba,X: (
        -(k3n*V + k4n + k_6n)*Di + Da
        + (k6n*P*X*(gsk0-(1+K8n*Ba)*Da-Di-Db))/(K21+w*X)
    )
    dDbdtn = lambda t,V,Di,Db,Bp,Da,P,Ba,X: k9n*Da*Ba - k10n*Db
    dBpdtn = lambda t,V,Di,Db,Bp,Da,P,Ba,X: k10n*Db - k11n*Bp

    # Matrix entries for implicit WNT block
    An = lambda t,V,Di,Db,Bp,Da,P,Ba,X: -K8n*(K8+w*Ba)*X/(K21+w*X)
    Bn = lambda t,V,Di,Db,Bp,Da,P,Ba,X: K7n*X
    Cn = lambda t,V,Di,Db,Bp,Da,P,Ba,X: K20n*X - w*K8n*Da*X/(K21+w*X)
    Dn = lambda t,V,Di,Db,Bp,Da,P,Ba,X: (
        1 + K7n*P + K20n*Ba
        + K21*w*(gsk0-(1+K8n*Ba)*Da-Di-Db)/(K21+w*X)**2
    )
    En = lambda t,V,Di,Db,Bp,Da,P,Ba,X: (1+K8n*Ba)
    Fn = lambda t,V,Di,Db,Bp,Da,P,Ba,X: K8n*Da
    Gn = lambda t,V,Di,Db,Bp,Da,P,Ba,X: K8n*Ba
    Hn = lambda t,V,Di,Db,Bp,Da,P,Ba,X: K17n*Ba
    # Constant-K16 In (no Ci)
    In_m = lambda t,V,Di,Db,Bp,Da,P,Ba,X: (
        1 + K8n*Da + (K16*tcf0)/(K16+w*Ba)**2 + K17n*P + K20n*X
    )
    Jn = lambda t,V,Di,Db,Bp,Da,P,Ba,X: K20n*Ba
    Kn = lambda t,V,Di,Db,Bp,Da,P,Ba,X: 1+K8n*Ba
    Ln = lambda t,V,Di,Db,Bp,Da,P,Ba,X: 1+K7n*X+K17n*Ba
    Mn = lambda t,V,Di,Db,Bp,Da,P,Ba,X: K8n*Da+K17n*P
    Nn = lambda t,V,Di,Db,Bp,Da,P,Ba,X: K7n*P

    RHS1n = lambda t,V,Di,Db,Bp,Da,P,Ba,X: (
        v14n + (k3n*V+k_6n)*Di
        - k6n*P*X*(gsk0-(1+K8n*Ba)*Da-Di-Db)/(K21+w*X)
        - k15n*P*X/(Km+w*P)
        + w*X/(K21+w*X)*(dDidtn(t,V,Di,Db,Bp,Da,P,Ba,X)+dDbdtn(t,V,Di,Db,Bp,Da,P,Ba,X))
    )
    RHS2n = lambda t,V,Di,Db,Bp,Da,P,Ba,X: k4n*Di-(1+k9n*Ba)*Da+k10n*Db
    RHS3n = lambda t,V,Di,Db,Bp,Da,P,Ba,X,H5,H13,Ci: (
        v12n - (k13n+k9n*Da+kp40n*H13+k41n*H5)*Ba + km40n*Ci
    )
    # Constant-K16 RHS4n (no Ci in K16 term)
    RHS4n = lambda t,V,Di,Db,Bp,Da,P,Ba,X,R,H13: (
        v18n/(1 + Ktn*Ba/(K16+w*Ba) + Kbn*Ba + gamma*w*H13)
        - APCdeg(R,P)*P
        - (dDidtn(t,V,Di,Db,Bp,Da,P,Ba,X)+dDbdtn(t,V,Di,Db,Bp,Da,P,Ba,X))
    )

    Matrixn = lambda t,V,Di,Db,Bp,Da,P,Ba,X: np.array([
        [An(t,V,Di,Db,Bp,Da,P,Ba,X), Bn(t,V,Di,Db,Bp,Da,P,Ba,X),
         Cn(t,V,Di,Db,Bp,Da,P,Ba,X), Dn(t,V,Di,Db,Bp,Da,P,Ba,X)],
        [En(t,V,Di,Db,Bp,Da,P,Ba,X), 0,
         Fn(t,V,Di,Db,Bp,Da,P,Ba,X), 0],
        [Gn(t,V,Di,Db,Bp,Da,P,Ba,X), Hn(t,V,Di,Db,Bp,Da,P,Ba,X),
         In_m(t,V,Di,Db,Bp,Da,P,Ba,X), Jn(t,V,Di,Db,Bp,Da,P,Ba,X)],
        [Kn(t,V,Di,Db,Bp,Da,P,Ba,X), Ln(t,V,Di,Db,Bp,Da,P,Ba,X),
         Mn(t,V,Di,Db,Bp,Da,P,Ba,X), Nn(t,V,Di,Db,Bp,Da,P,Ba,X)],
    ])
    Vectorn = lambda t,V,Di,Db,Bp,Da,P,Ba,X,R,H5,H13,Ci: np.array([
        RHS1n(t,V,Di,Db,Bp,Da,P,Ba,X),
        RHS2n(t,V,Di,Db,Bp,Da,P,Ba,X),
        RHS3n(t,V,Di,Db,Bp,Da,P,Ba,X,H5,H13,Ci),
        RHS4n(t,V,Di,Db,Bp,Da,P,Ba,X,R,H13),
    ])
    Solutionn = lambda t,V,Di,Db,Bp,Da,P,Ba,X,R,H5,H13,Ci: np.linalg.solve(
        Matrixn(t,V,Di,Db,Bp,Da,P,Ba,X),
        Vectorn(t,V,Di,Db,Bp,Da,P,Ba,X,R,H5,H13,Ci)
    )

    # RAS ODEs
    dRodtn = lambda t,Ro,Ra,A,R,B,Br,N,Nr,C,Cr,Dc,Dn,Bc: a*(rkm1*Ra-rkp1*Ro)+gtild(t)
    dRadtn = lambda t,Ro,Ra,A,R,B,Br,N,Nr,C,Cr,Dc,Dn,Bc: a*(rkp1*Ro-(rkm1+rk2*r*A)*Ra)
    dAdtn  = lambda t,Ro,Ra,A,R,B,Br,N,Nr,C,Cr,Dc,Dn,Bc: (a/r)*rv19 - a*(rk23*A)
    dRdtn  = lambda t,Ro,Ra,A,R,B,Br,N,Nr,C,Cr,Dc,Dn,Bc: (
        treat(t) + a*(rk2*r*Ra*A - rkp3*r*R*B + (rkm3+rk14)*Br
                      - rkp4*r*R*N + (rkm4+rk15)*Nr - rkp5*r*R*C + rkm5*Cr)
    )
    dBdtn  = lambda t,Ro,Ra,A,R,B,Br,N,Nr,C,Cr,Dc,Dn,Bc: (
        (a/r)*rv16 + a*(-(rk20+rkp3*r*R)*B + rkm3*Br + rk10*Dn)
    )
    dBrdtn = lambda t,Ro,Ra,A,R,B,Br,N,Nr,C,Cr,Dc,Dn,Bc: (
        a*(rkp3*r*R*B - rkm3*Br - rkp6*r*Br*C + rkm6*Dc
           - rkp9*r*Br*N + rkm9*Dn - rk14*Br)
    )
    dNdtn  = lambda t,Ro,Ra,A,R,B,Br,N,Nr,C,Cr,Dc,Dn,Bc: (
        (a/r)*rv17 + a*(-(rk21+rkp4*r*R)*N + rkm4*Nr - rkp9*r*Br*N + rkm9*Dn)
    )
    dNrdtn = lambda t,Ro,Ra,A,R,B,Br,N,Nr,C,Cr,Dc,Dn,Bc: (
        a*(rkp4*r*R*N - (rkm4+rk15)*Nr + rk10*Dn)
    )
    # dCdtn – no Ci in cyp_synthFunc for constant-K16 model
    dCdtn  = lambda t,Ro,Ra,A,R,B,Br,N,Nr,C,Cr,Dc,Dn,Bc,Ba: (
        (a/r)*cyp_synthFunc(R,Ba)
        + a*(-(rk22+rkp5*r*R)*C + (rkm5+rk8)*Cr - rkp6*r*Br*C + rkm6*Dc)
    )
    dCrdtn = lambda t,Ro,Ra,A,R,B,Br,N,Nr,C,Cr,Dc,Dn,Bc: a*(rkp5*r*R*C-(rkm5+rk8)*Cr)
    dDcdtn = lambda t,Ro,Ra,A,R,B,Br,N,Nr,C,Cr,Dc,Dn,Bc: a*(rkp6*r*Br*C-(rkm6+rk7)*Dc)
    dDndtn = lambda t,Ro,Ra,A,R,B,Br,N,Nr,C,Cr,Dc,Dn,Bc: a*(rkp9*r*Br*N-(rkm9+rk10)*Dn)
    dBcdtn = lambda t,Ro,Ra,A,R,B,Br,N,Nr,C,Cr,Dc,Dn,Bc: a*rk7*Dc

    k35n = lambda Ca: (k34 + a3*(w*Ca)**1) / (1+(phi*w*Ca)**1) / k5
    v30n = lambda bv: (Kc + a2*w*bv) / (1+w*bv) / (w*k5)

    dH5dtn  = lambda t,H5,M,Mi,Ca,H13,Ci,Nr,Ba: k31n+k32n*Mi+k33n(Nr)*Nr-k35n(Ca)*H5
    dMdtn   = lambda t,H5,M,Mi,Ca,H13,Ci,Nr,Ba: v30n(bctf(Ba))-k36n*M-kp37n*M*Mi+km37n*Ca
    dMidtn  = lambda t,H5,M,Mi,Ca,H13,Ci,Nr,Ba: v31n-k38n*Mi-kp37n*M*Mi+km37n*Ca
    dCadt   = lambda t,H5,M,Mi,Ca,H13,Ci,Nr,Ba: kp37n*M*Mi-km37n*Ca
    dH13dtn = lambda t,H5,M,Mi,Ca,H13,Ci,Nr,Ba: v32n-k39n*H13-kp40n*H13*Ba+km40n*Ci
    dCidtn  = lambda t,H5,M,Mi,Ca,H13,Ci,Nr,Ba: kp40n*H13*Ba-km40n*Ci

    def RHSn(t, U):
        return np.array([
            dRodtn (t,U[0],U[1],U[2],U[3],U[4],U[5],U[6],U[7],U[8],U[9],U[10],U[11],U[12]),
            dRadtn (t,U[0],U[1],U[2],U[3],U[4],U[5],U[6],U[7],U[8],U[9],U[10],U[11],U[12]),
            dAdtn  (t,U[0],U[1],U[2],U[3],U[4],U[5],U[6],U[7],U[8],U[9],U[10],U[11],U[12]),
            dRdtn  (t,U[0],U[1],U[2],U[3],U[4],U[5],U[6],U[7],U[8],U[9],U[10],U[11],U[12]),
            dBdtn  (t,U[0],U[1],U[2],U[3],U[4],U[5],U[6],U[7],U[8],U[9],U[10],U[11],U[12]),
            dBrdtn (t,U[0],U[1],U[2],U[3],U[4],U[5],U[6],U[7],U[8],U[9],U[10],U[11],U[12]),
            dNdtn  (t,U[0],U[1],U[2],U[3],U[4],U[5],U[6],U[7],U[8],U[9],U[10],U[11],U[12]),
            dNrdtn (t,U[0],U[1],U[2],U[3],U[4],U[5],U[6],U[7],U[8],U[9],U[10],U[11],U[12]),
            dCdtn  (t,U[0],U[1],U[2],U[3],U[4],U[5],U[6],U[7],U[8],U[9],U[10],U[11],U[12],U[19]),
            dCrdtn (t,U[0],U[1],U[2],U[3],U[4],U[5],U[6],U[7],U[8],U[9],U[10],U[11],U[12]),
            dDcdtn (t,U[0],U[1],U[2],U[3],U[4],U[5],U[6],U[7],U[8],U[9],U[10],U[11],U[12]),
            dDndtn (t,U[0],U[1],U[2],U[3],U[4],U[5],U[6],U[7],U[8],U[9],U[10],U[11],U[12]),
            dBcdtn (t,U[0],U[1],U[2],U[3],U[4],U[5],U[6],U[7],U[8],U[9],U[10],U[11],U[12]),
            dVdtn  (t,U[13],U[14],U[15],U[16],U[17],U[18],U[19],U[20]),
            dDidtn (t,U[13],U[14],U[15],U[16],U[17],U[18],U[19],U[20]),
            dDbdtn (t,U[13],U[14],U[15],U[16],U[17],U[18],U[19],U[20]),
            dBpdtn (t,U[13],U[14],U[15],U[16],U[17],U[18],U[19],U[20]),
            *Solutionn(t,U[13],U[14],U[15],U[16],U[17],U[18],U[19],U[20],U[3],U[21],U[25],U[26]),
            dH5dtn (t,U[21],U[22],U[23],U[24],U[25],U[26],U[7],U[19]),
            dMdtn  (t,U[21],U[22],U[23],U[24],U[25],U[26],U[7],U[19]),
            dMidtn (t,U[21],U[22],U[23],U[24],U[25],U[26],U[7],U[19]),
            dCadt  (t,U[21],U[22],U[23],U[24],U[25],U[26],U[7],U[19]),
            dH13dtn(t,U[21],U[22],U[23],U[24],U[25],U[26],U[7],U[19]),
            dCidtn (t,U[21],U[22],U[23],U[24],U[25],U[26],U[7],U[19]),
        ])

    # Initial conditions
    V_0=0; Di_0=4.83e-3; Db_0=2.02e-3; Bp_0=1
    Da_0=9.66e-3; P_0=18.116; Ba_0=25.1; X_0=4.93e-4
    W_0n = (1/w)*np.array([V_0,Di_0,Db_0,Bp_0,Da_0,P_0,Ba_0,X_0])

    R_0n = (1/1000)*np.array([10,10,10,100,1,0,1,0,0.01,0,0,0,0])

    H5_0=0.00146; M_0=0.04999; Mi_0=0.0601
    Ca_0=0.84028; H13_0=0.04439; Ci_0=0.0012
    H_0n = np.array([H5_0,M_0,Mi_0,Ca_0,H13_0,Ci_0])

    U_0n = np.concatenate([R_0n, W_0n, H_0n])
    sol = solve_ivp(RHSn, (t_0,t_N), U_0n, method="BDF", t_eval=time)
    print(sol.success, sol.message)
    t = sol.t; U = sol.y.T
    BcatTcf = bctf(U[:,19])
    Sol = np.column_stack([U, BcatTcf])
    return t, Sol, treatvals


In [ ]:
p = np.array([
    0.5, 0.1, 0.5, (2.13e11)*(1e-9), 1.32e1, (6.00e9)*(1e-0), 3.60e1,
    (1.2e11)*(1e-9), 4.00e1, 1.01e1, 1.01e1, 1.02e1, 2.70e1,
    (3.60e9)*(1e-7), 3.00e1, 1.00e-1, 3.85e-2, 1.73e-3, 2.00e-1,
    2.00e-1, 2.00e-2, 2.50e-2, 3.85e-2, 1.73e-1, 8.35e-3, 1.00e-2, 1.00e-1,
])

retef=0.0015; wntef=100; CCret=1000.0; CCwnt=1000.0
funcpercent=1.0; k8log=False; k17log=False
gamma1=1.0; apc_s=1

print("Running constant-K16 model...")
t1, Sol1, _ = modelsys_2025(p, retef, wntef, CCret, CCwnt, funcpercent, apc_s, k8log, k17log)

print("\nRunning dynamic-K16 model...")
t2, Sol2, _ = modelsys_2025_k16dynamic(p, retef, wntef, CCret, CCwnt, funcpercent, k8log, k17log, gamma1)

print("\nSol1 shape:", Sol1.shape, "| Sol2 shape:", Sol2.shape)


In [ ]:
# Constant K16 (Sol1, t1)
H5_const  = Sol1[:, 21]; H13_const = Sol1[:, 25]
M_const   = Sol1[:, 22]; Mi_const  = Sol1[:, 23]
Ca_const  = Sol1[:, 24]; Ci_const  = Sol1[:, 26]
Ba_const  = Sol1[:, 19]; P_const   = Sol1[:, 18]
Bp_const  = Sol1[:, 16]; BcatTcf_const = Sol1[:, 27]

# Dynamic K16 (Sol2, t2)
H5_dyn  = Sol2[:, 21]; H13_dyn = Sol2[:, 25]
M_dyn   = Sol2[:, 22]; Mi_dyn  = Sol2[:, 23]
Ca_dyn  = Sol2[:, 24]; Ci_dyn  = Sol2[:, 26]
Ba_dyn  = Sol2[:, 19]; P_dyn   = Sol2[:, 18]
Bp_dyn  = Sol2[:, 16]; BcatTcf_dyn = Sol2[:, 27]

mask1 = (t1 >= 0)
mask2 = (t2 >= 0)

c_black = (0.1, 0.1, 0.1)
c_red   = (0.6, 0.0, 0.0)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
pairs = [
    (axes[0,0], H5_const,  H5_dyn,  r"HOXA5"),
    (axes[0,1], H13_const, H13_dyn, r"HOXA13"),
    (axes[1,0], M_const,   M_dyn,   r"MYC"),
    (axes[1,1], Mi_const,  Mi_dyn,  r"MIZ1"),
]
for ax, yc, yd, title in pairs:
    ax.plot(t1[mask1], yc[mask1], color=c_black, linewidth=2.5, label=r"$K_{16}$ constant")
    ax.plot(t1[mask1], yd[mask1], color=c_red,   linewidth=2.5, label=r"$K_{16}$ dynamic")
    ax.set_title(title, fontsize=22)
    ax.set_xlabel(r"time ($\tau$)", fontsize=20)
    ax.set_ylabel("(non-dim)", fontsize=20)
    ax.tick_params(labelsize=20)
    ax.grid(False); ax.set_frame_on(True)

axes[1,1].legend(fontsize=16, loc="best")
fig.suptitle(r"Comparison of constant vs dynamic $K_{16}$ on HOX pathway components",
             fontsize=20, fontweight="bold")
fig.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
pairs = [
    (axes[0,0], P_const,       P_dyn,       r"APC"),
    (axes[0,1], Ba_const,      Ba_dyn,      r"$\beta$-catenin"),
    (axes[1,0], BcatTcf_const, BcatTcf_dyn, r"$\beta$-cat:TCF"),
    (axes[1,1], Bp_const,      Bp_dyn,      r"Phospho-$\beta$-cat"),
]
for ax, yc, yd, title in pairs:
    ax.plot(t1[mask1], yc[mask1], color=c_black, linewidth=2.5, label=r"$K_{16}$ constant")
    ax.plot(t2[mask2], yd[mask2], color=c_red,   linewidth=2.5, label=r"$K_{16}$ dynamic")
    ax.set_title(title, fontsize=22)
    ax.set_xlabel(r"time ($\tau$)", fontsize=20)
    ax.set_ylabel("(non-dim)", fontsize=20)
    ax.tick_params(labelsize=20)
    ax.set_frame_on(True)

axes[1,1].legend(fontsize=18, loc="best")
fig.suptitle(r"Comparison of constant vs dynamic $K_{16}$ on WNT pathway components",
             fontsize=20, fontweight="bold")
fig.tight_layout()
plt.show()


In [ ]:
shade_color = (0.96, 0.96, 0.96)
shade_start = 3000; shade_end = 25000

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
wnt_vars = [
    (axes[0,0], P_dyn,       r"APC ($P$)"),
    (axes[0,1], Ba_dyn,      r"$\beta$-catenin ($B_a$)"),
    (axes[1,0], BcatTcf_dyn, r"$\beta$-cat:TCF ($B_t$)"),
    (axes[1,1], Bp_dyn,      r"Phospho-$\beta$-cat ($B_p$)"),
]
for ax, ydata, title in wnt_vars:
    ax.plot(t2[mask2], ydata[mask2], color=c_black, linewidth=2.5, zorder=2)
    ax.axvspan(shade_start, shade_end, color=shade_color, zorder=0)
    ax.set_title(title, fontsize=20)
    ax.set_xlabel(r"time ($\tau$)", fontsize=18)
    ax.set_ylabel("(non-dim)", fontsize=18)
    ax.tick_params(labelsize=17)
    ax.set_frame_on(True)

fig.suptitle(r"Dynamics of WNT Pathway Components under Transient WNT Input",
             fontsize=20, fontweight="bold")
fig.tight_layout()
plt.show()


In [ ]:
shade_color = (0.95, 0.95, 0.95)
shade_start = 3000; shade_end = 25000

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
hox_const = [
    (axes[0,0], H5_const,  r"HOXA5 ($H_a$)"),
    (axes[0,1], H13_const, r"HOXA13 ($H_i$)"),
    (axes[0,2], M_const,   r"MYC ($M$)"),
    (axes[1,0], Mi_const,  r"MIZ1 ($M_i$)"),
    (axes[1,1], Ca_const,  r"MYC:MIZ1 ($C_a$)"),
    (axes[1,2], Ci_const,  r"HOXA13:$\beta$-cat ($C_i$)"),
]
for ax, ydata, title in hox_const:
    y = ydata[mask1]
    ax.plot(t1[mask1], y, color=c_black, linewidth=2.5, zorder=2)
    ax.axvspan(shade_start, shade_end, color=shade_color, zorder=0)
    ax.set_title(title, fontsize=15)
    ax.set_xlabel(r"time ($\tau$)", fontsize=13)
    ax.set_ylabel("(non-dim)", fontsize=13)
    ax.tick_params(labelsize=13)
    ax.set_frame_on(True)

fig.suptitle(r"HOX Components under Constant $K_{16}$",
             fontsize=20, fontweight="bold")
fig.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
hox_dyn = [
    (axes[0,0], H5_dyn,  r"HOXA5 ($H_a$)"),
    (axes[0,1], H13_dyn, r"HOXA13 ($H_i$)"),
    (axes[0,2], M_dyn,   r"MYC ($M$)"),
    (axes[1,0], Mi_dyn,  r"MIZ1 ($M_i$)"),
    (axes[1,1], Ca_dyn,  r"MYC:MIZ1 ($C_a$)"),
    (axes[1,2], Ci_dyn,  r"HOXA13:$\beta$-cat ($C_i$)"),
]
for ax, ydata, title in hox_dyn:
    y = ydata[mask2]
    ymin = y.min(); ymax = y.max(); pad = 0.05*(ymax-ymin) if ymax > ymin else 1e-9
    ax.plot(t2[mask2], y, color=c_red, linewidth=2.5, zorder=2)
    ax.axvspan(shade_start, shade_end,
               ymin=0, ymax=1, color=(0.95,0.95,0.95), zorder=0)
    ax.set_title(title, fontsize=15)
    ax.set_xlabel(r"time ($\tau$)", fontsize=13)
    ax.set_ylabel("(non-dim)", fontsize=13)
    ax.tick_params(labelsize=13)
    ax.set_frame_on(True)

fig.suptitle(r"Dynamic Response of HOX Components to Transient WNT Activation",
             fontsize=20, fontweight="bold")
fig.tight_layout()
plt.show()
